# CLIP-EBC ONNX 추론 튜토리얼

이 노트북은 `src/onnx` 모듈을 사용하여 군중 계수를 수행하는 방법을 보여줍니다.

**실행 전 주의사항:**
- 프로젝트 루트 디렉토리(`CLIP_EBC_ONNX/`)에서 Jupyter를 실행하세요
- `conda activate ebc` 환경을 사용하세요

## 0. 경로 설정

노트북이 `notebooks/` 안에 있으므로 프로젝트 루트를 기준으로 경로를 맞춥니다.

In [ ]:
import os
import sys

# notebooks/ 안에서 실행되므로 프로젝트 루트로 이동
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(project_root)
sys.path.insert(0, project_root)

assert os.path.exists('src/onnx/model.py'), f'프로젝트 루트에서 실행하세요. 현재: {os.getcwd()}'
print(f'작업 디렉토리: {os.getcwd()}')

## 1. 모델 로드

In [ ]:
from src.config import InferenceConfig
from src.onnx import ClipEBCOnnx

# ONNX 모델 자동 다운로드
import assets

config = InferenceConfig(model_path='assets/CLIP_EBC_nwpu_rmse_onnx.onnx')
model = ClipEBCOnnx(config)
print('모델 로드 완료')

## 2. 단일 이미지 추론

In [ ]:
# 파일 경로로 추론
count = model.predict_single('assets/289.jpg')
print(f'예측 군중 수: {count:.2f}명')

In [ ]:
# numpy 배열로 추론
import numpy as np
from PIL import Image

img = np.array(Image.open('assets/289.jpg').convert('RGB'))
print(f'이미지 shape: {img.shape}, dtype: {img.dtype}')

count = model.predict_single(img)
print(f'예측 군중 수: {count:.2f}명')

## 3. 배치 추론 (여러 이미지 한번에)

In [ ]:
# 여러 이미지를 리스트로 넣으면 한번에 추론
img1 = np.array(Image.open('assets/289.jpg').convert('RGB'))
img2 = np.array(Image.open('assets/sample.png').convert('RGB'))
img3 = np.array(Image.open('assets/sample2.png').convert('RGB'))

counts = model.predict([img1, img2, img3])

for name, count in zip(['289.jpg', 'sample.png', 'sample2.png'], counts):
    print(f'{name}: {count:.2f}명')

## 4. 시각화 (predict_dense_dot)

`predict_dense_dot()`을 사용하면 추론 + 히트맵 + 점 오버레이를 한번에 얻을 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt

def show_results(model, image_path, alpha=0.5):
    """predict_dense_dot으로 추론 + 시각화."""
    img = np.array(Image.open(image_path).convert('RGB'))
    counts, heats, dots = model.predict_dense_dot([img], alpha=alpha)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(img)
    axes[0].set_title('원본')
    axes[0].axis('off')

    axes[1].imshow(heats[0])
    axes[1].set_title(f'밀도 맵 (count={counts[0]:.1f})')
    axes[1].axis('off')

    axes[2].imshow(dots[0])
    axes[2].set_title(f'위치 탐지 ({int(round(counts[0]))}명)')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    return counts[0]

In [ ]:
show_results(model, 'assets/289.jpg')

In [ ]:
show_results(model, 'assets/sample.png')

In [ ]:
show_results(model, 'assets/sample2.png')

## 5. 속도 측정

In [ ]:
import time

img = np.array(Image.open('assets/289.jpg').convert('RGB'))

# warmup
for _ in range(3):
    model.predict([img])

# 측정
N = 20
times = []
for _ in range(N):
    t = time.perf_counter()
    model.predict([img])
    times.append((time.perf_counter() - t) * 1000)

print(f'ONNX 추론 속도 ({N}회 평균):')
print(f'  평균: {np.mean(times):.1f}ms')
print(f'  최소: {np.min(times):.1f}ms')
print(f'  이미지: {img.shape[1]}x{img.shape[0]}')

## 6. Config 커스터마이징

윈도우 크기, 스트라이드 등을 조정할 수 있습니다.

In [ ]:
from src.config import InferenceConfig

# 기본 설정 확인
config = InferenceConfig(model_path='assets/CLIP_EBC_nwpu_rmse_onnx.onnx')
print(f'window_size: {config.window_size}')
print(f'stride: {config.stride}')
print(f'reduction: {config.reduction}')
print(f'mean: {config.mean}')
print(f'std: {config.std}')